# **Scraping Planos Ibi Telecom**

_Página do provedor: https://ibitelecom.com.br/internet-fibra-otica-para-voce_

In [6]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date
import re

## **1. Extração**

In [7]:
# Obter lista de planos
texto_html = open("../../data/scraped/ibi.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find_all("div", class_="owl-stage")[0]
planos = container_planos.find_all("div", class_="owl-item cloned")
print(f"{len(planos)} cartões de planos capturados.")

6 cartões de planos capturados.


In [8]:
# Extrair informações dos planos
def raspar_plano(plano):
    titulo = plano.find("h2").get_text(strip=True)
    preco = plano.find("div", class_="card-plano__price").get_text(strip=True)
    try:
        preco_depois = plano.find("div", class_="card-plano__promo").get_text(strip=True)
    except:
        preco_depois = None
    return {
        "titulo": titulo,
        "preco": preco,
        "preco_depois": preco_depois
    }

dados = [raspar_plano(p) for p in planos]
df = pd.DataFrame(dados)
df = df.drop_duplicates()

df

,titulo,preco,preco_depois
0,900 mega + Prime Video ou HBO Max ou Disney+ (...,"R$ 125,00/mês",NaN
1,Contrate 480 e leve 800 MEGA + apps,"R$ 99,90/mês",NaN
2,850 mega + Globoplay Padrão + Apps,"R$99,90*por 6 meses","R$ 120,00"


## **2. Tratamento**

In [9]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Ibi"

# Velocidade
df_tratado["VELOCIDADE"] = (
    df_tratado["titulo"]
    .astype(str)
    .str.extract(r"(\d+)\s*(?=mega|mb)", flags=re.IGNORECASE)[0]
    .astype("Int64")
)

# PRECO
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .astype(str)
    .str.replace(",", ".", regex=False)
    .str.extract(r"(\d+\.?\d*)")[0]
    .astype(float)
)

# SVAs
df_tratado["SVAs"] = (
    df_tratado["titulo"]
    .astype(str)
    .str.split("+", n=1)
    .str[1]
    .where(lambda x: x.notna(), pd.NA)
    .str.strip()
)

# Periodo
df_tratado["PERIODO"] = df_tratado["preco"].str.extract(r'(\d+)\s*meses').astype(float)

# Preco depois
df_tratado["PRECO_DEPOIS"] = (
    df_tratado["preco_depois"]
    .str.extract(r'([\d\.,]+)')[0]
    .str.replace('.', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO", "PERIODO", "PRECO_DEPOIS"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO,PERIODO,PRECO_DEPOIS
0,2026-05-12,Ibi,900,Prime Video ou HBO Max ou Disney+ (apps com an...,125.0,NaN,NaN
1,2026-05-12,Ibi,800,apps,99.9,NaN,NaN
2,2026-05-12,Ibi,850,Globoplay Padrão + Apps,99.9,6.0,120.0


## **3. Carga**

In [10]:
# Salvando
provedor = "IBI.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-IBI.csv salvo!
